# 10. Futility-анализ (ранняя остановка по бесперспективности)


## 0. Мотивация: зачем останавливаться по бесперспективности

В прошлой лекции мы останавливали эксперимент **раньше**, когда эффект уже доказан — это *efficacy-остановка*: статистика $Z_t$ пересекла верхнюю границу, и мы уверены в эффекте. Сегодня — «обратная сторона»: остановить тест, когда эффекта, скорее всего, **нет или он меньше MDE**. Это *futility-остановка*.

**Почему это вообще нужно.** Трафик и время аналитиков — ограниченный ресурс. Тест, который «застрял» на слабом эффекте, продолжает жечь трафик до финального горизонта, а этот же трафик мог бы уже тестировать следующую гипотезу. Задержка каждого «мёртвого» теста — это отложенный релиз других, потенциально успешных, изменений (opportunity cost). Futility-анализ — формализованный ответ на вопрос:

> Если на промежуточном взгляде эффект выглядит слабее MDE, есть ли смысл тратить оставшийся трафик?

**Ключевая асимметрия, которую мы будем держать в голове.** Остановка по эффективности и остановка по бесперспективности — это *разные* риски. Peeking по эффективности раздувает ошибку I рода (мы чаще видим «эффект» там, где его нет) — поэтому её надо контролировать α-spending. А остановка по futility, как мы увидим в разделе 7, **не раздувает ошибку I рода** — она лишь жертвует мощностью. Это делает futility «безопасной» остановкой, и именно поэтому её так любят в продуктовых A/B.

**План лекции.** Сначала введём общий аппарат — стандартизованную статистику $Z_t$ и долю информации $t$ (раздел 1). Потом два способа оценить «шанс дожить до значимости»: частотный *Conditional Power* (раздел 2) и байесовский *Predictive Power* (раздел 3). Объединим их рамкой *stochastic curtailment* и переведём порог в futility-границу (раздел 4). Затем построим заранее фиксированную futility-границу через *β-spending* — зеркало α-spending (раздел 5), разберём влияние на ошибки I/II рода (разделы 6–7) и сведём всё в практический чек-лист (раздел 8).


In [ ]:
import math
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import norm
from scipy.optimize import brentq
from ipywidgets import interact, FloatSlider, IntSlider, Dropdown

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.labelsize"] = 12

SEED = 12345
rng = np.random.default_rng(SEED)

# Единая палитра исходов на всю лекцию
C_FUT = "#d95f02"   # оранжевый — futility (бесперспективность)
C_EFF = "#1b9e77"   # зелёный — efficacy (эффект есть)
C_CONT = "#999999"  # серый — продолжаем


In [ ]:
@dataclass(frozen=True)
class Design:
    """Параметры сквозного примера: A/B-тест непрерывной метрики (средний чек)."""
    alpha: float = 0.05        # односторонняя ошибка I рода
    beta: float = 0.20         # ошибка II рода (power = 0.80)
    sigma: float = 100.0       # СКО метрики, руб.
    mu0: float = 500.0         # среднее в контроле
    mde: float = 10.0          # абсолютный MDE, руб.
    K: int = 5
    t: tuple = (0.30, 0.50, 0.70, 0.85, 1.0)  # доли информации (моменты взглядов)

    @property
    def delta(self) -> float:
        """Стандартизованный MDE: delta = mde / sigma."""
        return self.mde / self.sigma

    @property
    def theta_mde(self) -> float:
        """Дрейф (нецентральность) под MDE: theta = z_{1-alpha} + z_{1-beta}."""
        return norm.ppf(1 - self.alpha) + norm.ppf(1 - self.beta)

    @property
    def i_max(self) -> float:
        """Максимальная информация: I_max = (theta_mde / delta)^2."""
        return (self.theta_mde / self.delta) ** 2

    @property
    def n_max(self) -> float:
        """Максимальный размер выборки на группу (двухвыборочный z-тест)."""
        return 2 * self.sigma ** 2 * self.i_max


design = Design()
design


### 0.1. Сквозной пример и «цена» продолжения теста

Через всю лекцию пройдёт один пример, чтобы все формулы и графики были про одно и то же. Метрика — **средний чек** (непрерывная величина со стандартным отклонением $\sigma = 100$ руб.), в контроле $\mu_0 = 500$ руб. Мы хотим поймать подъём на $MDE = 10$ руб. с мощностью 80% при одностороннем $\alpha = 5\%$.

Смысл таблицы ниже — почувствовать масштаб: полный тест «стоит» десятки миллионов показов, и каждый процент ранней остановки — это реально сэкономленный трафик, который можно отдать следующему эксперименту.


In [ ]:
n = int(np.ceil(design.n_max))
print(f"Стандартизованный MDE:  delta = {design.delta:.3f}")
print(f"Дрейф под MDE:          theta_mde = {design.theta_mde:.3f}")
print(f"Размер на группу (макс): n_max = {n:,} пользователей")
print(f"Суммарный трафик теста:  {2*n:,} пользователей")

cost = pd.DataFrame({
    "Момент остановки": ["25%", "50%", "75%", "100%"],
    "Доля трафика": [0.25, 0.50, 0.75, 1.00],
    "Пользователей": [int(2 * n * x) for x in (0.25, 0.50, 0.75, 1.00)],
})
cost["Сэкономлено vs 100%"] = (2 * n - cost["Пользователей"]).astype(int)
cost


## 1. Аппарат: информация, дрейф и B-value

Весь futility-анализ строится на одной идее: любая метрика и любой момент времени сводятся к **одной стандартизованной величине** $Z_t$. Это удобно, потому что тогда все формулы ниже не зависят от того, что именно мы меряем — чек, конверсию, время на странице. Разберём, откуда берётся $Z_t$ и почему он устроен так просто.

### 1.1. Вывод распределения $Z_t$

Пусть есть две независимые выборки объёма $n_t$ каждая: контроль $A$ и тест $B$, из нормальных распределений с общей дисперсией $\sigma^2$. Оценка разницы средних $\hat\delta_t = \bar X_B - \bar X_A$ имеет дисперсию, равную сумме дисперсий двух средних:

$$\mathrm{Var}(\hat\delta_t) = \frac{\sigma^2}{n_t} + \frac{\sigma^2}{n_t} = \frac{2\sigma^2}{n_t}.$$

Стандартизуем — поделим на стандартную ошибку $SE_t = \sigma\sqrt{2/n_t}$:

$$Z_t = \frac{\hat\delta_t}{SE_t} \;\sim\; \mathcal{N}\!\left(\frac{\delta}{SE_t},\;1\right),$$

где $\delta = \mu_B - \mu_A$ — истинный эффект. **Смысл стандартизации**: какая бы ни была метрика, $Z_t$ уже безразмерен и имеет дисперсию ровно 1 — это «линейка», по которой можно сравнивать любые тесты.

Теперь введём **информацию** $I_t = 1/SE_t^2 = n_t/(2\sigma^2)$ — это обратная дисперсия оценки, то есть мера того, «сколько данных мы накопили». Тогда математическое ожидание:

$$\mathbb{E}[Z_t] = \frac{\delta}{SE_t} = \delta\sqrt{I_t} = \underbrace{\delta\sqrt{I_{max}}}_{=\;\theta}\cdot\sqrt{\frac{I_t}{I_{max}}} = \theta\sqrt{t}.$$

Здесь появились две важные величины:

- $t = I_t/I_{max} \approx n_t/n_{max}$ — **доля информации**, «сколько теста уже позади» (от 0 до 1);
- $\theta = \delta\sqrt{I_{max}}$ — **дрейф**, «сила сигнала» в единицах стандартизованной статистики на финальном анализе.

Итог компактный и универсальный:

$$Z_t \sim \mathcal{N}\big(\theta\sqrt{t},\; 1\big).$$

### 1.2. Независимые приращения и B-value

Главное свойство, которое нам понадобится для расчёта границ: статистика накапливается по мере прихода данных, и **новые данные не знают о старых**. Формально: приращения на непересекающихся интервалах информации независимы, поэтому ковариация двух моментов $s \le t$ пропорциональна доле общей информации:

$$\mathrm{Cov}(Z_s, Z_t) = \sqrt{s/t}.$$

Чтобы этим было удобно пользоваться, переходят к «броуновской» величине $B(t) = \sqrt{t}\,Z_t$. У неё математическое ожидание $\theta t$, дисперсия $t$, и — решающее — **независимые приращения**:

$$B(t) = \sqrt{t}\,Z_t \;\sim\; \mathcal{N}(\theta t,\; t), \qquad B(t) - B(s) \perp\!\!\!\perp B(s).$$

**К чему это ведёт.** Дальше вся работа — это «наблюдаем случайное блуждание $B(t)$ с дрейфом $\theta$ и решаем, когда остановиться». Независимость приращений позволит в разделе 5 численно сворачивать плотность «не остановившихся» траекторий и точно находить границы.


In [ ]:
def z_stat(xA: np.ndarray, xB: np.ndarray, sigma: float) -> float:
    """Двухвыборочный z-тест непрерывной метрики (известная дисперсия)."""
    nA, nB = xA.size, xB.size
    se = sigma * np.sqrt(1 / nA + 1 / nB)
    return (xB.mean() - xA.mean()) / se


def simulate_b(rng, theta: float, t: np.ndarray, n_sim: int = 200) -> np.ndarray:
    """Траектории B(t) = sqrt(t)*Z_t — броуновское движение с дрейфом theta.

    Каждый шаг генерирует независимое нормальное приращение
    B(t_k) - B(t_{k-1}) ~ N(theta * dt, dt),  dt = t_k - t_{k-1}.
    """
    dt = np.diff(np.concatenate(([0.0], t)))
    inc = rng.normal(loc=theta * dt, scale=np.sqrt(dt), size=(n_sim, len(t)))
    return np.cumsum(inc, axis=1)


In [ ]:
# Проверка формулы Z_t ~ N(theta*sqrt(t), 1) на симуляции:
# убеждаемся, что среднее и дисперсия статистики совпадают с теорией.
t = np.asarray(design.t)
Z = simulate_b(rng, design.theta_mde, t, 20000) / np.sqrt(t)
print("Эмпирическое среднее Z_t под H1:", np.round(Z.mean(axis=0), 3))
print("Теория  theta*sqrt(t):           ", np.round(design.theta_mde * np.sqrt(t), 3))
print("Эмпирическое SD (должно быть ~1):", np.round(Z.std(axis=0), 3))


In [ ]:
# График нужен, чтобы "увидеть" дрейф: под H0 траектории топчутся у нуля,
# под H1 — систематически уходят вверх со скоростью theta*t.
fig, axes = plt.subplots(1, 2, figsize=(12, 4.4), sharey=True)

cases = [(0.0, "H0: эффекта нет (θ = 0)", "траектории топчутся около нуля"),
         (design.theta_mde, "H1: эффект = MDE", "траектории уходят вверх с дрейфом θt")]

for ax, (theta, title, note) in zip(axes, cases):
    B = simulate_b(rng, theta, t, 150)
    ax.plot(t, B.T, color=C_CONT, alpha=0.25, lw=0.9)
    ax.plot(t, theta * t, color="black", lw=2.2, ls="--", label=f"средний дрейф θt = {theta:.2f}t")
    ax.set_title(f"{title}\n{note}", fontsize=12)
    ax.set_xlabel("доля информации t")
    ax.legend(loc="upper left")

axes[0].set_ylabel("B(t) = √t · Z_t")
fig.suptitle("Броуновское движение с дрейфом — это и есть наш процесс наблюдений", fontsize=13.5)
fig.tight_layout()
plt.show()


## 2. Conditional Power (CP): условная вероятность «добить» тест

Теперь ответим на главный вопрос futility: *«если сейчас эффект выглядит слабо, с какой вероятностью тест всё-таки дойдёт до значимости к концу?»*. Ответ на этот вопрос называется **conditional power** — вероятность отвергнуть $H_0$ в конце при условии текущих данных и при **заданном** предположении об истинном эффекте $\theta$.

**Почему это правильная величина для решения о futility.** Решение «продолжать или остановиться» — это ставка на будущее: «стоит ли тратить оставшийся трафик?». CP ровно это и измеряет: шанс, что будущие данные доведут тест до значимости. Если шанс мал — тратить трафик бессмысленно.

### 2.1. Вывод формулы

Ключевой трюк — выразить будущее через настоящее и независимое приращение. На шкале $B$ финальная статистика раскладывается так:

$$Z_1 = B(1) = B(t) + \big(B(1) - B(t)\big) = \sqrt{t}\,z_t + \big(B(1) - B(t)\big).$$

Первое слагаемое мы уже **наблюдали** ($\sqrt t\, z_t$), а второе — **независимое** приращение, распределённое как $\mathcal{N}(\theta(1-t),\; 1-t)$ (по разделу 1.2). Поэтому условно на $Z_t = z_t$ будущее распределение известно:

$$Z_1 \mid Z_t = z_t \;\sim\; \mathcal{N}\big(\sqrt{t}\,z_t + \theta(1-t),\;\; 1-t\big).$$

Вероятность попасть в критическую область $\{Z_1 > z_{1-\alpha}\}$ — это просто хвост этого нормального распределения:

$$\mathrm{CP}(\theta) = P_\theta(Z_1 > z_{1-\alpha} \mid z_t)
= 1 - \Phi\!\left(\frac{z_{1-\alpha} - \sqrt{t}\,z_t - \theta(1-t)}{\sqrt{1-t}}\right).$$

**Что говорит структура формулы.** Числитель — «насколько далеко финальная граница от ожидаемого финального значения», знаменатель — «сколько ещё неопределённости осталось». Обратите внимание на два крайних случая: при $t \to 0$ (данных почти нет) CP стремится к априорной мощности дизайна; при $t \to 1$ (тест почти закончен) CP превращается в ступеньку — судьба уже решена, и данные почти не могут её изменить. Это важно для выбора *когда* смотреть на futility (см. раздел 8).

Правило futility: **если $\mathrm{CP} < \gamma$ — останавливаемся** (типично $\gamma \in [0.10, 0.20]$).

### 2.2. Какое $\theta$ подставить?

Формула CP зависит от выбранного $\theta$ — и это самый тонкий момент. Три осмысленных варианта:

- $\theta = 0$ (под $H_0$): «даже если эффекта нет, дойдём ли мы до ложной значимости?» — консервативно для futility, редко останавливает.
- $\theta = \hat\theta_t = z_t/\sqrt t$ (текущий тренд): «считаем, что будущее похоже на то, что мы уже видим». Заманчиво, но **оптимистично**: текущая оценка шумная, и на ранних взглядах она завышает шансы (эффект, который «выстрелил» случайно, кажется многообещающим).
- $\theta = \theta_{MDE}$ (дизайновый эффект): «если эффект ровно такой, как мы закладывали в power-расчёте». Это стандартный ориентир — он отвечает на вопрос «есть ли шанс поймать тот эффект, ради которого мы затевали тест».

Разницу между ними показывает график ниже.


In [ ]:
def conditional_power(z_t: float, t: float, theta: float, z_alpha: float = 1.6448536) -> float:
    """CP: вероятность Z_1 > z_alpha при наблюдаемом z_t и истинном дрейфе theta.

    Формула: 1 - Phi( (z_alpha - sqrt(t)*z_t - theta*(1-t)) / sqrt(1-t) ).
    """
    return 1.0 - norm.cdf((z_alpha - np.sqrt(t) * z_t - theta * (1 - t)) / np.sqrt(1 - t))


In [ ]:
z_alpha = norm.ppf(1 - design.alpha)
z_grid = np.linspace(-3, 3, 300)
tt = 0.5   # середина теста

fig, ax = plt.subplots(figsize=(9, 5))

# три предположения об эффекте
ax.plot(z_grid, conditional_power(z_grid, tt, 0.0, z_alpha),
        color=C_CONT, lw=2.2, label="CP под H0 (θ = 0) — консервативно")
ax.plot(z_grid, conditional_power(z_grid, tt, design.theta_mde, z_alpha),
        color=C_EFF, lw=2.2, label="CP под MDE (θ = θ_MDE) — дизайновый ориентир")
ax.plot(z_grid, conditional_power(z_grid, tt, z_grid / np.sqrt(tt), z_alpha),
        color=C_FUT, lw=2.2, label="CP под трендом (θ = z_t/√t) — оптимистично")

# порог futility и зона остановки
ax.axhline(0.15, ls="--", color="black", lw=1.2)
ax.text(2.6, 0.16, "порог γ = 0.15", ha="right", fontsize=11)
ax.fill_between(z_grid, 0, 0.15, color=C_FUT, alpha=0.08)

# точка, где CP(MDE) пересекает порог
z_cut = brentq(lambda z: conditional_power(z, tt, design.theta_mde, z_alpha) - 0.15, -3, 3)
ax.axvline(z_cut, color=C_FUT, ls=":", lw=1.3)
ax.annotate(f"при z_t < {z_cut:.2f} → стоп (CP<0.15)",
            xy=(z_cut, 0.05), xytext=(z_cut - 2.0, 0.30),
            arrowprops=dict(arrowstyle="->", color=C_FUT), color=C_FUT, fontsize=11)

ax.set_xlabel("наблюдаемая статистика $z_t$")
ax.set_ylabel("conditional power")
ax.set_title(f"CP при t = {tt}: выбор θ кардинально меняет ответ")
ax.legend(loc="upper left")
plt.show()


### 2.3. Карта CP: как шансы меняются с течением теста

График выше — это срез карты CP в одной точке $t$. Полезно посмотреть на всю карту на плоскости $(t, z)$: она наглядно показывает, **почему время взгляда так важно**.

На ранних взглядах карта почти плоская — впереди ещё много данных, и почти любое текущее $z$ даёт средние шансы (судьбу теста решает будущее, а не прошлое). К концу карта «заостряется» в ступеньку — судьба почти решена, и данные уже мало что могут изменить. Контур $\mathrm{CP} = \gamma$ — это и есть граница «стоп / продолжаем», которую мы затем переведём на шкалу $z$ в разделе 4.


In [ ]:
T, Zz = np.meshgrid(np.linspace(0.05, 0.95, 120), np.linspace(-3, 3, 240))
CP = conditional_power(Zz, T, design.theta_mde, z_alpha)

fig, ax = plt.subplots(figsize=(9, 5.5))
cs = ax.contourf(T, Zz, CP, levels=30, cmap="viridis")
ct = ax.contour(T, Zz, CP, levels=[0.15], colors="red", linewidths=2.5)
ax.clabel(ct, fmt={0.15: "CP = 0.15"}, fontsize=11)

ax.text(0.75, -2.2, "СТОП (futility)\nCP < 0.15", color="white", fontsize=12, ha="center", weight="bold")
ax.text(0.75, 2.2, "продолжаем", color="black", fontsize=12, ha="center", weight="bold")

ax.set_xlabel("доля информации t")
ax.set_ylabel("наблюдаемая статистика $z_t$")
ax.set_title("Conditional power под MDE: красный контур — граница futility-остановки")
cbar = fig.colorbar(cs, ax=ax)
cbar.set_label("conditional power")
plt.show()


In [ ]:
def plot_cp_interactive():
    def _plot(t, z_t, theta, alpha):
        za = norm.ppf(1 - alpha)
        cp = conditional_power(z_t, t, theta, za)
        fig, ax = plt.subplots(figsize=(8, 2.6))
        ax.barh(0, cp, color=C_FUT if cp < 0.15 else C_EFF)
        ax.set_xlim(0, 1); ax.set_yticks([])
        ax.axvline(0.15, ls="--", color="black")
        ax.set_title(f"CP = {cp:.3f}  →  {'СТОП (futility)' if cp < 0.15 else 'продолжаем'}")
        plt.show()
    interact(_plot,
             t=FloatSlider(value=0.3, min=0.05, max=0.95, step=0.05, description="t"),
             z_t=FloatSlider(value=0.5, min=-3, max=3, step=0.05, description="z_t"),
             theta=FloatSlider(value=design.theta_mde, min=0, max=4, step=0.1, description="θ"),
             alpha=FloatSlider(value=0.05, min=0.01, max=0.20, step=0.01, description="α"))
plot_cp_interactive()


## 3. Predictive Power (PPoS): усредняем CP по неопределённости эффекта

Главная претензия к CP — жёсткая фиксация $\theta$. Когда мы подставляем «текущий тренд» $\hat\theta_t$, мы молча притворяемся, что шумная оценка — это и есть истина. Но на раннем взгляде $\hat\theta_t$ очень неточна: эффект, который «выстрелил» случайно, даёт обнадёживающе высокий CP. **Predictive power (PPoS)** снимает эту проблему, усредняя CP по всей апостериорной неопределённости эффекта:

$$\mathrm{PPoS} = \int \mathrm{CP}(\theta)\,\pi(\theta \mid D_t)\,d\theta.$$

Интуиция: вместо «каковы шансы при одном фиксированном θ» мы спрашиваем «каковы шансы в среднем по всем правдоподобным θ, взвешенным тем, насколько они вероятны после наших данных».

### 3.1. Вывод апостериорного распределения

Зададим априорное $\theta \sim \mathcal{N}(\theta_0, \tau^2)$: $\theta_0$ — наше исходное представление о типичном эффекте, $\tau$ — насколько мы в нём уверены (малое $\tau$ = сильный prior). Правдоподобие данных: $Z_t \mid \theta \sim \mathcal{N}(\theta\sqrt{t}, 1)$, что эквивалентно наблюдению $\hat y = z_t/\sqrt{t}$ с шумом $\mathcal{N}(\theta, 1/t)$. Для нормального prior и нормального правдоподобия апостериор снова нормален (сопряжённость), причём точность (обратная дисперсия) просто **складывается** из точности prior и точности данных:

$$\frac{1}{\sigma_p^2} = \frac{1}{\tau^2} + t, \qquad
\mu_p = \sigma_p^2\left(\frac{\theta_0}{\tau^2} + \sqrt{t}\, z_t\right).$$

Среднее $\mu_p$ — это взвешенная сумма prior $\theta_0$ и данных $\hat y$, где веса — точности. Чем больше данных (больше $t$), тем сильнее апостериор «прижимается» к данным; чем сильнее prior (меньше $\tau$), тем он дольше «сопротивляется».

### 3.2. Замкнутая форма PPoS

Подставим CP в интеграл. Числитель CP зависит от $\theta$ линейно: $z_{1-\alpha} - \sqrt{t}z_t - \theta(1-t) = A - W(1-t)$, где $A = z_{1-\alpha} - \sqrt{t}z_t - \mu_p(1-t)$ и $W = \theta - \mu_p \sim \mathcal{N}(0, \sigma_p^2)$. Используем тождество $\mathbb{E}\big[\Phi\big((a - W)/b\big)\big] = \Phi\big(a/\sqrt{b^2 + \sigma_W^2}\big)$ при $W \sim \mathcal{N}(0, \sigma_W^2)$ — оно как раз «размывает» CP апостериорной дисперсией:

$$\mathrm{PPoS} = 1 - \Phi\!\left(\frac{z_{1-\alpha} - \sqrt{t}\,z_t - \mu_p(1-t)}
{\sqrt{(1-t) + (1-t)^2\sigma_p^2}}\right).$$

**Что это значит.** Знаменатель $\sqrt{(1-t) + (1-t)^2\sigma_p^2}$ больше, чем у CP ($\sqrt{1-t}$), ровно на слагаемое, отвечающее за неопределённость эффекта. Поэтому PPoS **ниже** CP-под-трендом: он честно учитывает, что эффект мы знаем неточно. CP — частный случай PPoS с вырожденным (точечным) prior $\sigma_p^2 \to 0$. Для общения с бизнесом PPoS удобнее: это непосредственно «вероятность успеха в процентах».


In [ ]:
def ppos(z_t: float, t: float, theta0: float, tau: float, z_alpha: float = 1.6448536) -> float:
    """Predictive probability of success (нормальный prior, замкнутая форма Spiegelhalter–Freedman–Parmar)."""
    sigma2 = 1.0 / (1.0 / tau**2 + t)              # апостериорная дисперсия
    mu = sigma2 * (theta0 / tau**2 + np.sqrt(t) * z_t)   # апостериорное среднее
    return 1.0 - norm.cdf((z_alpha - np.sqrt(t) * z_t - mu * (1 - t))
                          / np.sqrt((1 - t) + (1 - t)**2 * sigma2))


In [ ]:
z_t, tt = 0.4, 0.5
theta0, tau = 0.0, 0.8            # скептический prior: эффект в среднем = 0
sigma2 = 1.0 / (1.0 / tau**2 + tt)
mu = sigma2 * (theta0 / tau**2 + np.sqrt(tt) * z_t)

theta_grid = np.linspace(-0.5, 4, 400)
post = norm.pdf(theta_grid, loc=mu, scale=np.sqrt(sigma2))          # апостериор
cp_theta = conditional_power(z_t, tt, theta_grid, z_alpha)          # CP(θ)
pp = ppos(z_t, tt, theta0, tau, z_alpha)                            # PPoS
cp_trend = conditional_power(z_t, tt, z_t / np.sqrt(tt), z_alpha)   # CP по тренду

fig, axes = plt.subplots(1, 2, figsize=(12, 4.6), gridspec_kw={"width_ratios": [1, 1.35]})

# левая панель — апостериор эффекта
axes[0].plot(theta_grid, post, color=C_CONT, lw=2)
axes[0].fill_between(theta_grid, post, color=C_CONT, alpha=0.15)
axes[0].axvline(mu, color="black", ls="--", lw=1.2, label=f"апостериорное среднее {mu:.2f}")
axes[0].axvline(design.theta_mde, color=C_EFF, ls=":", lw=1.2, label=f"MDE-дрейф {design.theta_mde:.2f}")
axes[0].set_title("Апостериор эффекта θ | данные\n(скептический prior N(0, 0.8²))")
axes[0].set_xlabel("θ"); axes[0].set_ylabel("плотность"); axes[0].legend(fontsize=9)

# правая панель — CP(θ) и интеграл PPoS
axes[1].plot(theta_grid, cp_theta, color=C_EFF, lw=2.2, label="CP(θ)")
axes[1].fill_between(theta_grid, cp_theta * post, alpha=0.35, color=C_FUT,
                     label="подынтегральное CP(θ)·π(θ)")
axes[1].axhline(pp, color=C_FUT, ls="--", lw=1.6, label=f"PPoS = {pp:.3f}")
axes[1].axhline(cp_trend, color=C_CONT, ls="--", lw=1.6, label=f"CP(тренд) = {cp_trend:.3f}")
axes[1].set_title("PPoS = среднее CP(θ) по апостериору")
axes[1].set_xlabel("θ"); axes[1].set_ylabel("вероятность"); axes[1].legend(fontsize=9)

fig.suptitle("PPoS «притянут» к массе апостериора у нуля → ниже, чем CP(тренд)", fontsize=13.5)
fig.tight_layout()
plt.show()


In [ ]:
def plot_ppos_interactive():
    def _plot(t, z_t, theta0, tau):
        pp = ppos(z_t, t, theta0, tau, z_alpha)
        cpt = conditional_power(z_t, t, z_t / np.sqrt(t), z_alpha)
        fig, ax = plt.subplots(figsize=(8, 2.6))
        ax.barh([1, 0], [pp, cpt], color=[C_FUT, C_CONT])
        ax.set_yticks([0, 1]); ax.set_yticklabels(["CP(тренд)", "PPoS"])
        ax.set_xlim(0, 1); ax.axvline(0.15, ls="--", color="black")
        ax.set_title(f"PPoS = {pp:.3f}   vs   CP(тренд) = {cpt:.3f}")
        plt.show()
    interact(_plot,
             t=FloatSlider(value=0.5, min=0.05, max=0.95, step=0.05, description="t"),
             z_t=FloatSlider(value=0.4, min=-3, max=3, step=0.05, description="z_t"),
             theta0=FloatSlider(value=0.0, min=-1, max=3, step=0.1, description="prior θ0"),
             tau=FloatSlider(value=0.8, min=0.05, max=3, step=0.05, description="prior τ"))
plot_ppos_interactive()


## 4. Stochastic Curtailment: общая рамка и перевод CP в границу

CP и PPoS — две внешне разные формулы, но за ними стоит одна общая идея — **stochastic curtailment** (Lan–Simon–Halperin, 1982): останавливаем тест, когда условная вероятность «полезного» финального исхода при некоем предположении становится пренебрежимо малой. CP — это частотная инкарнация (условная вероятность при фиксированном θ), PPoS — байесовская (усреднение по апостериору).

**Полезный взгляд на CP как на границу.** Порог $\mathrm{CP} < \gamma$ можно переписать в терминах статистики: при фиксированных $t, \alpha, \gamma, \theta$ неравенство $\mathrm{CP} < \gamma$ эквивалентно $z_t < a_{CP}$, где $a_{CP}$ — решение уравнения

$$\mathrm{CP}(a_{CP};\, t, \theta) = \gamma.$$

Иначе говоря, на плоскости $(t, z)$ правило «стоп по CP» — это просто кривая, ниже которой мы останавливаемся (мы видели её красный контур на карте в разделе 2.3). Решаем уравнение численно: CP монотонно растёт по $z_t$, поэтому корень единственный.

**Почему важно различать типы границ.** Такая CP-граница **данные-зависимая**: её положение зависит от того, что мы наблюдаем ($z_t$). А есть принципиально другой способ — задать futility-границу **заранее**, до сбора данных, чтобы она гарантировала контроль ошибки II рода. Этим способом (β-spending) мы займёмся в следующем разделе. Различие «граница по ходу теста» vs «граница, зафиксированная до» — одно из центральных в futility-анализе.


In [ ]:
def cp_to_boundary(t: float, gamma: float, theta: float, z_alpha: float = 1.6448536) -> float:
    """z_t, при котором CP(z_t; t, theta) == gamma (численная инверсия монотонной CP)."""
    f = lambda z: conditional_power(z, t, theta, z_alpha) - gamma
    return brentq(f, -8, 8)


In [ ]:
tt_grid = np.linspace(0.05, 0.95, 200)
gamma = 0.15

fig, ax = plt.subplots(figsize=(9, 5))

a_null = [cp_to_boundary(t_, gamma, 0.0, z_alpha) for t_ in tt_grid]
a_mde = [cp_to_boundary(t_, gamma, design.theta_mde, z_alpha) for t_ in tt_grid]

ax.plot(tt_grid, a_null, color=C_CONT, lw=2, label="граница по CP под H0 (θ = 0)")
ax.plot(tt_grid, a_mde, color=C_FUT, lw=2.2, label="граница по CP под MDE")
ax.fill_between(tt_grid, -4, a_mde, color=C_FUT, alpha=0.10)
ax.text(0.6, -1.4, "ниже границы — СТОП (futility)", color=C_FUT, fontsize=11)

# пример траектории, пересекающей границу
z_ex = np.array([1.6, 1.1, 0.55, 0.3])
t_ex = np.array([0.2, 0.5, 0.8, 0.95])
ax.plot(t_ex, z_ex, color="black", lw=1.6, marker="o",
        label="пример наблюдаемой статистики (заходит в зону futility)")

ax.set_xlabel("доля информации t")
ax.set_ylabel("futility-граница $a(t)$ на шкале z")
ax.set_title("Futility-граница из порога CP = 0.15 (данные-зависимая)")
ax.legend(loc="lower left")
plt.show()


## 5. β-spending: заранее фиксированная futility-граница

В прошлом разделе CP давал границу, которая «плавает» вместе с данными. Минус такой границы: мы не можем заранее гарантировать, сколько ошибки II рода (или I рода) она «съест». Альтернатива — построить futility-границу **до начала теста**, как мы это делаем с efficacy-границей. Инструмент — **β-spending**, точное зеркало α-spending.

**Логика симметрии.** В прошлой лекции α-spending отвечал на вопрос: «как распределить бюджет ошибки I рода α по промежуточным взглядам, чтобы в сумме ложное отвержение случалось с вероятностью не больше α?». β-spending задаёт тот же вопрос про ошибку II рода: «как распределить бюджет β по взглядам, чтобы в сумме вероятность *зря остановить тест, когда эффект на самом деле есть* была не больше β?». Заметьте тонкость: «зря остановить» возможно только когда эффект реально есть, поэтому futility-границу считают **под альтернативой** $H_1$ ($\theta = \theta_{MDE}$), тогда как efficacy-границу — под $H_0$.

### 5.1. Спендинг-функции

Оба бюджета «тратятся» неубывающими функциями от доли информации $t$:

- **α-spending** (Lan–DeMets): $\alpha(t)$, $\alpha(0)=0$, $\alpha(1)=\alpha$. Две классические формы — O'Brien–Fleming $\alpha(t)=2-2\Phi(z_{1-\alpha/2}/\sqrt t)$ (бережёт α в начале: граница высокая и почти не тратится) и Pocock $\alpha(t)=\alpha\ln(1+(e-1)t)$ (тратит α равномернее).
- **β-spending** (Kim–DeMets / Pampallona–Tsiatis): $\beta(t)$, $\beta(0)=0$, $\beta(1)=\beta$. Степенное семейство $\beta(t)=\beta\,t^{\rho}$: $\rho=3$ даёт OBF-подобную futility-границу (консервативно в начале — редко останавливаем рано), $\rho=1$ — Pocock-подобную (агрессивно тратим β рано).

### 5.2. Рекурсивный расчёт границ

Как из спендинг-функции получить числа $c_k$ (efficacy) и $a_k$ (futility)? Идея — в независимых приращениях $B$-процесса из раздела 1.2. Мы держим **плотность продолжения** — распределение траекторий, которые «дожили» до взгляда $k-1$, не пересекнув ни одну границу. Тогда:

- на взгляде $k$ efficacy-граница $c_k$ отсекает такой **правый хвост** этой плотности, чтобы добавленная вероятность пересечения равнялась приращению $\alpha(t_k)-\alpha(t_{k-1})$;
- futility-граница $a_k$ отсекает такую **левую часть**, чтобы добавленная вероятность futility-остановки под $H_1$ равнялась $\beta(t_k)-\beta(t_{k-1})$.

На каждом шаге плотность «урезается» до области продолжения и сворачивается с плотностью следующего приращения. Это точный численный аналог того, что Jennison–Turnbull называют «recursive numerical integration». Код ниже реализует ровно это.


In [ ]:
def alpha_spending(t, alpha: float = 0.05, family: str = "obf") -> np.ndarray:
    """Кумулятивная α-spending функция (Lan–DeMets)."""
    t = np.asarray(t, dtype=float)
    if family == "obf":
        return 2 - 2 * norm.cdf(norm.ppf(1 - alpha / 2) / np.sqrt(t))
    if family == "pocock":
        return alpha * np.log(1 + (np.e - 1) * t)
    raise ValueError(family)


def beta_spending(t, beta: float = 0.20, family: str = "obf") -> np.ndarray:
    """Кумулятивная β-spending функция (степенное семейство Kim–DeMets)."""
    t = np.asarray(t, dtype=float)
    if family == "obf":
        return beta * t ** 3          # ρ=3 — OBF-подобная
    if family == "pocock":
        return beta * t               # ρ=1 — Pocock-подобная
    raise ValueError(family)


In [ ]:
def _convolve(p: np.ndarray, grid: np.ndarray, loc: float, scale: float) -> np.ndarray:
    """Свёртка плотности с нормальным приращением N(loc, scale^2).

    p_next(x_j) = Σ_i p(x_i) * φ(x_j; x_i + loc, scale) * dx.
    """
    dx = grid[1] - grid[0]
    K = norm.pdf(grid[:, None], loc=grid[None, :] + loc, scale=scale)
    return (K @ p) * dx


def _quantile(p: np.ndarray, grid: np.ndarray, target: float, tail: bool) -> float:
    """Точка, отсекающая массу target: правый хвост (tail=True) или левую часть (tail=False).

    Для субплотности с общей массой cdf[-1] правый хвост = cdf[-1] - cdf(x).
    """
    dx = grid[1] - grid[0]
    cdf = np.cumsum(p) * dx
    vals = (cdf[-1] - cdf) if tail else cdf
    idx = int(np.argmin(np.abs(vals - target)))
    return grid[idx]


def group_sequential_boundaries(t, alpha=0.05, beta=0.20, theta=None,
                                fam_alpha="obf", fam_beta="obf",
                                grid_dx=0.01, n_sigma=6.0):
    """Односторонние efficacy (верхняя c) и futility (нижняя a) границы на шкале Z.

    Efficacy-граница считается под H0 (θ=0) без нижней границы (non-binding);
    futility-граница — под H1 (θ=θ_mde) с обеими границами в области продолжения.
    """
    t = np.asarray(t, dtype=float)
    K = len(t)
    if theta is None:
        theta = norm.ppf(1 - alpha) + norm.ppf(1 - beta)
    dt = np.diff(np.concatenate(([0.0], t)))
    grid = np.arange(-n_sigma, n_sigma + grid_dx, grid_dx)

    d_alpha = np.diff(np.concatenate(([0.0], alpha_spending(t, alpha, fam_alpha))))
    d_beta = np.diff(np.concatenate(([0.0], beta_spending(t, beta, fam_beta))))

    c = np.empty(K); a = np.empty(K)
    p_eff = norm.pdf(grid, loc=0.0, scale=math.sqrt(dt[0]))            # плотность B_1 под H0
    p_fut = norm.pdf(grid, loc=theta * dt[0], scale=math.sqrt(dt[0]))  # плотность B_1 под H1
    b_prev, a_prev = np.inf, -np.inf
    for k in range(K):
        # efficacy: правый хвост = приращение α
        b_k = _quantile(p_eff, grid, d_alpha[k], tail=True)
        c[k] = b_k / math.sqrt(t[k]); b_prev = b_k
        # futility: левая часть = приращение β
        a_k = _quantile(p_fut, grid, d_beta[k], tail=False)
        a[k] = a_k / math.sqrt(t[k]); a_prev = a_k
        if k < K - 1:
            d = dt[k + 1]
            # продолжаем только внутри области: efficacy — x ≤ b_k; futility — a_k < x ≤ b_k
            p_eff = _convolve(p_eff * (grid <= b_k), grid, 0.0, math.sqrt(d))
            p_fut = _convolve(p_fut * ((grid > a_k) & (grid <= b_k)), grid, theta * d, math.sqrt(d))
    return c, a


c_bound, a_bound = group_sequential_boundaries(design.t, design.alpha, design.beta, design.theta_mde)
boundaries = pd.DataFrame({
    "взгляд k": np.arange(1, design.K + 1),
    "t_k": design.t,
    "c_k (efficacy)": np.round(c_bound, 3),
    "a_k (futility)": np.round(a_bound, 3),
})
boundaries


In [ ]:
tt = np.asarray(design.t)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.4))

# α-spending
a_sp = alpha_spending(tt, design.alpha, "obf")
axes[0].plot(tt, a_sp, marker="o", color=C_EFF, lw=2.2)
axes[0].fill_between(tt, a_sp, color=C_EFF, alpha=0.18)
axes[0].axhline(design.alpha, ls="--", color="black", lw=1)
axes[0].text(0.02, design.alpha + 0.006, "бюджет α = 0.05", fontsize=10)
axes[0].set_title("α-spending (O'Brien–Fleming)\nна efficacy-границу")
axes[0].set_xlabel("t"); axes[0].set_ylabel("накопленная ошибка I рода")

# β-spending
b_sp = beta_spending(tt, design.beta, "obf")
axes[1].plot(tt, b_sp, marker="o", color=C_FUT, lw=2.2)
axes[1].fill_between(tt, b_sp, color=C_FUT, alpha=0.18)
axes[1].axhline(design.beta, ls="--", color="black", lw=1)
axes[1].text(0.02, design.beta + 0.006, "бюджет β = 0.20", fontsize=10)
axes[1].set_title("β-spending (ρ = 3)\nна futility-границу")
axes[1].set_xlabel("t"); axes[1].set_ylabel("накопленная ошибка II рода")

fig.suptitle("Две стороны одной монеты: как «тратится» ошибка I и II рода по ходу теста", fontsize=13.5)
fig.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))
tt = np.asarray(design.t)

ax.plot(tt, c_bound, color=C_EFF, lw=2.2, marker="o", label="efficacy-граница $c_k$")
ax.plot(tt, a_bound, color=C_FUT, lw=2.2, marker="o", label="futility-граница $a_k$")

# зоны
ax.fill_between(tt, c_bound, 4.2, color=C_EFF, alpha=0.12)
ax.fill_between(tt, -4.2, a_bound, color=C_FUT, alpha=0.12)
ax.fill_between(tt, a_bound, c_bound, color=C_CONT, alpha=0.06)

ax.text(0.66, 3.4, "СТОП — эффект\n(Z > c_k)", color=C_EFF, fontsize=12, ha="center", weight="bold")
ax.text(0.66, -2.2, "СТОП — futility\n(Z < a_k)", color=C_FUT, fontsize=12, ha="center", weight="bold")
ax.text(0.66, 0.8, "продолжаем", color="black", fontsize=12, ha="center")

ax.axhline(0, color="black", lw=0.8)
ax.set_xlabel("доля информации t")
ax.set_ylabel("Z(t)")
ax.set_ylim(-4.2, 4.2)
ax.set_title("Двухграничный групповой-последовательный дизайн")
ax.legend(loc="upper right")
plt.show()


### 5.3. Расширение на двустороннюю проверку

До сих пор мы рассматривали односторонний тест, где futility означает «эффект не догонит MDE в выгодную сторону». Для **двустороннего** теста (когда нас интересует отклонение в любую сторону) всё зеркально:

- **efficacy**: останавливаемся при $|Z_t| > c_k$ — граница симметрична, а бюджет α делится на два хвоста (по $\alpha/2$ на каждую сторону);
- **futility**: останавливаемся, когда эффект «зажат» в симметричном коридоре $|Z_t| < a_k$ — то есть величина эффекта слишком мала, чтобы дотянуться до значимости в любую из сторон.

Геометрически это та же картинка, отражённая ниже нуля: зона futility — «воронка» вокруг нуля, зоны efficacy — два симметричных хвоста. Смысл не меняется: futility по-прежнему означает «эффект, скорее всего, меньше того, что мы закладывали».


In [ ]:
z_alpha2 = norm.ppf(1 - design.alpha / 2)
theta2 = z_alpha2 + norm.ppf(1 - design.beta)   # дрейф для двустороннего MDE
c2, a2 = group_sequential_boundaries(design.t, design.alpha, design.beta, theta2)

fig, ax = plt.subplots(figsize=(9, 5.5))
tt = np.asarray(design.t)

ax.plot(tt, c2, color=C_EFF, lw=2.2, marker="o", label="± efficacy (двусторонняя)")
ax.plot(tt, -c2, color=C_EFF, lw=2.2, marker="o")
ax.plot(tt, a2, color=C_FUT, lw=2.2, marker="o", label="± futility (двусторонняя)")
ax.plot(tt, -a2, color=C_FUT, lw=2.2, marker="o")

ax.fill_between(tt, a2, -a2, color=C_FUT, alpha=0.12)
ax.fill_between(tt, c2, 4.2, color=C_EFF, alpha=0.12)
ax.fill_between(tt, -4.2, -c2, color=C_EFF, alpha=0.12)

ax.text(0.66, 3.4, "эффект (|Z| > c_k)", color=C_EFF, fontsize=12, ha="center", weight="bold")
ax.text(0.66, 0.0, "futility (|Z| < a_k)", color=C_FUT, fontsize=12, ha="center", weight="bold")
ax.axhline(0, color="black", lw=0.8)
ax.set_xlabel("доля информации t"); ax.set_ylabel("Z(t)")
ax.set_ylim(-4.2, 4.2)
ax.set_title("Двусторонний дизайн: futility = коридор |Z| < a_k вокруг нуля")
ax.legend(loc="upper right")
plt.show()


## 6. Binding vs non-binding futility

Раз futility-граница влияет на то, какие траектории «доживают» до конца, возникает тонкий вопрос: **как её учитывать при расчёте ошибки I рода α?** Отсюда два соглашения.

- **Binding (обязывающая)**: futility-стоп обязателен, и α считается с учётом того, что траектории ниже $a_k$ гарантированно останавливаются. Если же потом проигнорировать стоп (продолжить тест вопреки futility), часть «отброшенных» траекторий вернётся и суммарный α вырастет.
- **Non-binding (необязывающая)**: α считается так, *будто futility-границы нет*. Тогда решение «остановиться или нет» можно отменить в любой момент без риска для α.

**Почему non-binding — стандарт.** В реальности futility — это *рекомендация*, а не приговор: аналитик может увидеть, что порог сработал из-за шумного первого взгляда, и продолжить тест. Non-binding соглашение формально разрешает такую гибкость, не ломая контроль ошибки I рода. Именно поэтому в продуктовых A/B почти всегда используют non-binding.

Демонстрация ниже считает суммарный α efficacy-границы в двух случаях и показывает, что non-binding не превышает номинал.


In [ ]:
def overall_alpha_upper(c: np.ndarray, t: np.ndarray, lower=None, n_sim: int = 500000):
    """Monte-Carlo: P(хотя бы один взгляд Z_k > c_k) под H0 (θ=0).

    lower=None  → futility-границы нет (non-binding, α считаем как без неё);
    lower=a_k   → binding: остановленные по futility исключаются.
    """
    dt = np.diff(np.concatenate(([0.0], t)))
    B = np.cumsum(rng.normal(0, np.sqrt(dt), size=(n_sim, len(t))), axis=1)
    Z = B / np.sqrt(t)
    active = np.ones(n_sim, dtype=bool)
    crossed = np.zeros(n_sim, dtype=bool)
    for k in range(len(t)):
        if lower is not None:                      # binding: убираем остановленных по futility
            active &= ~(Z[:, k] < lower[k]) & ~crossed
        crossed |= active & (Z[:, k] > c[k])
        active &= ~crossed
    return crossed.mean()


t = np.asarray(design.t)
alpha_none = overall_alpha_upper(c_bound, t, lower=None)
alpha_bind = overall_alpha_upper(c_bound, t, lower=a_bound)

res = pd.DataFrame({
    "Сценарий": ["без futility-границы (non-binding)", "binding"],
    "Суммарный α (efficacy)": np.round([alpha_none, alpha_bind], 4),
})
res["Отличие от номинального α = 0.05"] = np.round(res["Суммарный α (efficacy)"] - design.alpha, 4)
res


## 7. Влияние futility на ошибки I/II рода + Monte-Carlo

Теперь самый важный статистический факт лекции, ради которого всё и затевалось.

**Почему futility НЕ раздувает ошибку I рода.** Ошибка I рода — это ложное отвержение $H_0$: траектория, которая под нулевой гипотезой всё-таки забралась выше efficacy-границы. Futility-граница отбрасывает траектории **ниже** себя — те, что выглядят слабо. Такие траектории вряд ли добрались бы до efficacy-границы, а если бы и добрались, futility-стоп только *уменьшает* их число. Поэтому вероятность отвергнуть при $H_0$ может лишь **уменьшиться**, но не вырасти. Это радикально отличается от peeking по эффективности, который добавляет лишние «шансы» увидеть эффект.

**Почему futility снижает мощность.** А вот под $H_1$ (эффект реально есть) часть траекторий на раннем взгляде *случайно* выглядит слабо — и futility-граница их зря останавливает. Эти «неудачные заходы» — плата за экономию трафика: мы теряем часть мощности.

Проверим оба утверждения симуляцией и измерим «цену»: мощность, вероятность futility-стопа под H0/H1 и средний размер выборки (ASN — average sample number).


In [ ]:
def simulate_design(theta, c, a, t, n_sim=50000):
    """MC группо-последовательного дизайна. Исходы: efficacy / futility / complete."""
    dt = np.diff(np.concatenate(([0.0], t)))
    B = np.cumsum(rng.normal(theta * dt, np.sqrt(dt), size=(n_sim, len(t))), axis=1)
    Z = B / np.sqrt(t)
    outcome = np.full(n_sim, "complete")
    stop_frac = np.ones(n_sim)
    for k in range(len(t)):
        active = outcome == "complete"
        if not active.any():
            break
        eff = active & (Z[:, k] > c[k])
        fut = active & (Z[:, k] < a[k])
        outcome[eff] = "efficacy"; outcome[fut] = "futility"
        stop_frac[eff] = t[k]; stop_frac[fut] = t[k]
    return outcome, stop_frac


def oc(theta, c, a, t, n_sim=50000):
    """Операционные характеристики: (power, P(futility), P(complete), ASN-доля)."""
    out, stop = simulate_design(theta, c, a, t, n_sim)
    p_eff = (out == "efficacy").mean()
    p_fut = (out == "futility").mean()
    p_comp = (out == "complete").mean()
    return p_eff, p_fut, p_comp, stop.mean()


t = np.asarray(design.t)
p0 = oc(0.0, c_bound, a_bound, t)
p1 = oc(design.theta_mde, c_bound, a_bound, t)
print(f"Под H0 (θ=0):     FPR={p0[0]:.4f}  futility-stop={p0[1]:.3f}  complete={p0[2]:.3f}  E[N]/Nmax={p0[3]:.3f}")
print(f"Под H1 (θ=MDE):   power={p1[0]:.4f}  futility-stop={p1[1]:.3f}  complete={p1[2]:.3f}  E[N]/Nmax={p1[3]:.3f}")


In [ ]:
# Операционные характеристики по спектру эффектов + доверительные полосы MC.
# Смысл: увидеть «цену» futility сразу при всех возможных истинных эффектах.
thetas = np.linspace(-0.2, 3.0, 17)
N = 30000
rows = [oc(th, c_bound, a_bound, t, N) for th in thetas]
oc_df = pd.DataFrame(rows, columns=["power", "futility_stop", "complete", "asn"])
oc_df["theta"] = thetas
se = np.sqrt(oc_df[["power", "futility_stop", "complete"]].mul(1 - oc_df[["power", "futility_stop", "complete"]]) / N)

fig, ax = plt.subplots(figsize=(9, 5.2))
ax.plot(thetas, oc_df["power"], color=C_EFF, lw=2.2, label="power (дошли до efficacy)")
ax.plot(thetas, oc_df["futility_stop"], color=C_FUT, lw=2.2, label="P(stop по futility)")
ax.plot(thetas, oc_df["complete"], color=C_CONT, lw=2.2, label="P(дошли до конца)")
for col, color in [("power", C_EFF), ("futility_stop", C_FUT), ("complete", C_CONT)]:
    ax.fill_between(thetas, oc_df[col] - 2 * se[col], oc_df[col] + 2 * se[col],
                    color=color, alpha=0.12)

ax.axvline(design.theta_mde, color="black", ls="--", lw=1.3)
ax.text(design.theta_mde + 0.05, 0.9, "θ_MDE", fontsize=11)
ax.set_xlabel("истинный дрейф θ")
ax.set_ylabel("вероятность исхода")
ax.set_title("Операционные характеристики: сумма трёх кривых = 1")
ax.legend(loc="center right")
plt.show()


In [ ]:
# Ансамбль траекторий: наглядно видно, как futility-граница "отфильтровывает"
# бесперспективные заходы под H0 и изредка зря останавливает тест под H1.
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
cases = [(0.0, "H0: почти всё уходит в futility — это выгодная экономия"),
         (design.theta_mde, "H1: большинство доходит до efficacy, часть «застревает»")]

for ax, (theta, title) in zip(axes, cases):
    dt = np.diff(np.concatenate(([0.0], t)))
    B = np.cumsum(rng.normal(theta * dt, np.sqrt(dt), size=(250, len(t))), axis=1)
    Z = B / np.sqrt(t)
    for i in range(250):
        col = C_CONT
        for k in range(len(t)):
            if Z[i, k] > c_bound[k]:
                col = C_EFF; break
            if Z[i, k] < a_bound[k]:
                col = C_FUT; break
        ax.plot(t, Z[i], color=col, alpha=0.3, lw=0.8)
    ax.plot(t, c_bound, color=C_EFF, lw=2.4)
    ax.plot(t, a_bound, color=C_FUT, lw=2.4)
    ax.set_title(title, fontsize=12)
    ax.set_xlabel("t")

axes[0].set_ylabel("Z(t)")
from matplotlib.lines import Line2D
legend = [Line2D([0], [0], color=C_EFF, lw=2.4, label="efficacy"),
          Line2D([0], [0], color=C_FUT, lw=2.4, label="futility"),
          Line2D([0], [0], color=C_CONT, lw=2, alpha=0.5, label="дошли до конца")]
fig.legend(handles=legend, loc="lower center", ncol=3, fontsize=11)
fig.suptitle("Ансамбль траекторий, раскрашенный по исходу: видно случайное время остановки", fontsize=13.5)
fig.tight_layout(rect=[0, 0.06, 1, 0.96])
plt.show()


## 8. Практические рекомендации и связь с α-spending

Соберём всё в чек-лист дизайнера. Каждый пункт — не догма, а следствие того, что мы разобрали выше.

1. **Тайминг взглядов — не раньше 30–50% данных.** Ранние данные слишком шумные (вспомните: при малом $t$ карта CP почти плоская), поэтому ранний futility-стоп почти ничего не «знает» и зря теряет мощность. Поздние взгляды, наоборот, почти бесплатные — судьба теста и так ясна.
2. **Порог CP/PPoS — 10–20%.** Агрессивнее (выше 20%) — заметная потеря мощности; мягче (ниже 5%) — экономии почти нет. Это тот самый Парето-компромисс «мощность ↔ трафик».
3. **Выбор θ для CP.** Под $H_0$ — консервативно, под трендом — оптимистично, под MDE — дизайновый ориентир. PPoS — когда хотите честно учесть неопределённость эффекта.
4. **Non-binding — дефолт.** Futility-стоп можно отменить, не испортив контроль α.
5. **Полный дизайн = α-spending + β-spending.** Это две стороны одной монеты на оси $t$: α-бюджет строит efficacy-границу, β-бюджет — futility-границу. Вместе они дают полностью контролируемый по обеим ошибкам групповой-последовательный дизайн.

Интерактив ниже собирает всё вместе: крутите число взглядов и формы спендинга — и смотрите, как меняются границы и операционные характеристики.


In [ ]:
def design_full(alpha=0.05, beta=0.20, K=5, fam_alpha="obf", fam_beta="obf"):
    t = np.linspace(0.3, 1.0, K)
    theta = norm.ppf(1 - alpha) + norm.ppf(1 - beta)
    c, a = group_sequential_boundaries(t, alpha, beta, theta, fam_alpha, fam_beta)
    p0 = oc(0.0, c, a, t, 20000)
    p1 = oc(theta, c, a, t, 20000)
    return t, c, a, p0, p1


def plot_design_dashboard(K, fam_alpha, fam_beta):
    t, c, a, p0, p1 = design_full(0.05, 0.20, K, fam_alpha, fam_beta)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))
    axes[0].plot(t, c, color=C_EFF, lw=2.2, marker="o", label="efficacy")
    axes[0].plot(t, a, color=C_FUT, lw=2.2, marker="o", label="futility")
    axes[0].fill_between(t, a, c, color=C_CONT, alpha=0.06)
    axes[0].set_title("Границы"); axes[0].set_xlabel("t"); axes[0].legend()
    axes[1].bar(["FPR (H0)", "power (H1)", "futility (H1)"],
                [p0[0], p1[0], p1[1]], color=[C_CONT, C_EFF, C_FUT])
    axes[1].set_ylim(0, 1)
    axes[1].set_title(f"OC: E[N]/Nmax = {p1[3]:.2f} (под H1)")
    fig.tight_layout(); plt.show()

interact(plot_design_dashboard,
         K=IntSlider(value=3, min=2, max=8, step=1, description="взглядов K"),
         fam_alpha=Dropdown(options=["obf", "pocock"], value="obf", description="α-spending"),
         fam_beta=Dropdown(options=["obf", "pocock"], value="obf", description="β-spending"))


## 9. Итоги

1. **Futility не раздувает ошибку I рода** — в отличие от peeking по эффективности; она лишь снижает мощность.
2. **CP** отвечает «достигнем ли значимости при фиксированном θ», **PPoS** усредняет по неопределённости θ; CP — частный случай PPoS.
3. **Stochastic curtailment** — общая рамка; её инкарнации — CP и PPoS.
4. **β-spending — зеркало α-spending**: даёт априорную futility-границу с контролем ошибки II рода.
5. **Non-binding** — безопасный дефолт; тайминг/порог/prior определяют компромисс «мощность ↔ экономия трафика».

| | Conditional Power | PPoS | β-spending |
|---|---|---|---|
| Философия | частотная | байесовская | частотная (дизайн) |
| Предположение об эффекте | фиксированное θ | prior θ | θ_MDE |
| Граница | данные-зависимая | данные-зависимая | фиксирована заранее |
| Что контролирует | правило «стоп» | правило «стоп» | ошибку II рода |
| Когда | быстро оценить шансы | учесть неопределённость | спроектировать тест |
